In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time

## Batched causal linear attention

We keep the unnormalized recurrence: `S_t = S_(t-1) + outer(k_t, v_t)` and `o_t = q_t @ S_t`.

Inputs `q` and `k` have shape `[B, T, d_k]`; `v` has shape `[B, T, d_v]`.
`B` is the number of independent sequences and `T` is their length.
The outer loop picks one sequence. It starts a fresh memory matrix `S` of shape `[d_k, d_v]`.
The inner loop advances through that sequence using `torch.outer` and ordinary matrix multiplication.
We stack the token outputs, then stack the sequences to return `[B, T, d_v]`.

`torch.outer` takes two one-dimensional vectors, so we process sequences separately.
This is a readable learning implementation; the extra Python loop can be slower than batched operations.
For now we supply Q/K/V directly. Learned projections from an input `x` come later.


In [3]:
class CausalLinearAttention(nn.Module):
    def forward(self, q, k, v):
        # q, k: [B, T, d_k]; v: [B, T, d_v]
        B, T, d_k = q.shape
        d_v = v.shape[-1]
        batch_outputs = []

        for b in range(B):
            # each sequence starts with its own memory
            S = q.new_zeros(d_k, d_v)
            outputs = []

            for t in range(T):
                # [d_k] outer [d_v] -> [d_k, d_v]
                S = S + torch.outer(k[b, t], v[b, t])
                # [d_k] @ [d_k, d_v] -> [d_v]
                output_t = q[b, t] @ S
                outputs.append(output_t)

            batch_outputs.append(torch.stack(outputs))  # [T, d_v]

        return torch.stack(batch_outputs)  # [B, T, d_v]


In [4]:
torch.manual_seed(0)

B, T, d_k, d_v = 2, 4, 3, 2
# float64 makes the numerical comparison stricter
# requires_grad=True lets us check derivatives with respect to all inputs.
q = torch.randn(B, T, d_k, dtype=torch.float64, requires_grad=True)
k = torch.randn(B, T, d_k, dtype=torch.float64, requires_grad=True)
v = torch.randn(B, T, d_v, dtype=torch.float64, requires_grad=True)

attention = CausalLinearAttention()
o = attention(q, k, v)
print("Output shape:", o.shape)  # [2, 4, 2]


Output shape: torch.Size([2, 4, 2])


# Delta Net

In [10]:
class DeltaNet(nn.Module):
    # initialize learnable tensor weights with correct dims
    def __init__(self, d_model, layer_idx=None):
        super().__init__()
        # Zoology gives layer_idx, this attention variant does not need it
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.beta_proj = nn.Linear(d_model, 1, bias=False)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        # get batch size (n of sequences), sequence length and d_k (key dimension)
        B, T, d_k = x.shape

        q = F.normalize(F.silu(self.W_q(x)), dim=-1)
        k = F.normalize(F.silu(self.W_k(x)), dim=-1)
        v = self.W_v(x)
        # get d_v  (value dimension)
        d_v = v.shape[-1]
        beta = torch.sigmoid(self.beta_proj(x))  # [B, T, 1]

        batch_outputs = []

        for b in range(B):
            # each sequence starts with its own memory
            S = q.new_zeros(d_k, d_v)
            outputs = []

            for t in range(T):
                # read the memory prediction for current key
                v_hat = k[b, t] @ S      # [d_k] @ [d_k, d_v] -> [d_v]
                # calculate prediction error
                error = v[b, t] - v_hat
                # write a correction
                S = S + beta[b, t, 0] * torch.outer(k[b, t], error)
                
                output_t = q[b, t] @ S    # [d_k] @ [d_k, d_v] -> [d_v]
                outputs.append(output_t)

            batch_outputs.append(torch.stack(outputs))  # [T, d_v]
        # collect results [B, T, hidden_size]
        o = torch.stack(batch_outputs)
        # transform each token's result
        return self.out_proj(o)


In [11]:
# check the trainable DeltaNet layer: [B, T, d_model] -> [B, T, d_model].
torch.manual_seed(0)

B, T, d_model = 2, 4, 3
x = torch.randn(B, T, d_model)

delta_net = DeltaNet(d_model=d_model)
delta_output = delta_net(x)

assert delta_output.shape == (B, T, d_model)
print("DeltaNet output shape:", delta_output.shape)  # [2, 4, 3]


DeltaNet output shape: torch.Size([2, 4, 3])


# Gated Delta Net

In [ ]:
class GatedDeltaNet(nn.Module):
    # initialize learnable tensor weights with correct dims
    def __init__(self, d_model, layer_idx=None):
        super().__init__()
        # Zoology gives layer_idx, this attention variant does not need it
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.beta_proj = nn.Linear(d_model, 1, bias=False)
        self.alpha_proj = nn.Linear(d_model, 1)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        # get batch size (n of sequences), sequence length and d_k (key dimension)
        B, T, d_k = x.shape

        q = F.normalize(F.silu(self.W_q(x)), dim=-1)
        k = F.normalize(F.silu(self.W_k(x)), dim=-1)
        v = self.W_v(x)
        # get d_v  (value dimension)
        d_v = v.shape[-1]
        # assign value of beta and alpha for each token (alpha has simplified initialization)
        beta = torch.sigmoid(self.beta_proj(x))  # [B, T, 1]
        alpha = torch.sigmoid(self.alpha_proj(x))  # [B, T, 1]

        batch_outputs = []

        for b in range(B):
            # each sequence starts with its own memory
            S = q.new_zeros(d_k, d_v)
            outputs = []

            for t in range(T):
                # forget part of the previous memory
                S = alpha[b, t, 0] * S
                # read the memory prediction for current key
                v_hat = k[b, t] @ S      # [d_k] @ [d_k, d_v] -> [d_v]
                # calculate prediction error
                error = v[b, t] - v_hat
                # write a correction
                S = S + beta[b, t, 0] * torch.outer(k[b, t], error)
                
                output_t = q[b, t] @ S    # [d_k] @ [d_k, d_v] -> [d_v]
                outputs.append(output_t)

            batch_outputs.append(torch.stack(outputs))  # [T, d_v]
        # collect results [B, T, hidden_size]
        o = torch.stack(batch_outputs)
        # transform each token's result
        return self.out_proj(o)


In [7]:
# Check GatedDeltaNet: [B, T, d_model] -> [B, T, d_model].
torch.manual_seed(0)

B, T, d_model = 2, 4, 3
x = torch.randn(B, T, d_model)

gated_delta_net = GatedDeltaNet(d_model=d_model)
gated_delta_output = gated_delta_net(x)

assert gated_delta_output.shape == (B, T, d_model)
assert torch.isfinite(gated_delta_output).all()
print("GatedDeltaNet output shape:", gated_delta_output.shape)  # [2, 4, 3]


GatedDeltaNet output shape: torch.Size([2, 4, 3])
